In [2]:
import openai
import os
import sys
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm, trange

# Set your OpenAI API key



sys.path.append("../../")
import biked_commons
from biked_commons.resource_utils import split_datasets_path, models_and_scalers_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.design_evaluation import construct_tensor_evaluator, get_standard_evaluations
from biked_commons.transformation import one_hot_encoding

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
filedir = "openai_files"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [5]:
data = pd.read_csv(split_datasets_path("bike_bench_mixed_modality.csv"), index_col=0)
data_oh = one_hot_encoding.encode_to_continuous(data)
data_tens = torch.tensor(data_oh.values, dtype=torch.float32).to(device)

In [9]:
data["Number of cogs"]

1        9
2        0
3        0
4        9
5        0
        ..
4796     8
4797    10
4798     8
4799     8
4800     8
Name: Number of cogs, Length: 4499, dtype: int64

In [4]:
def get_condition_by_idx(idx=0):
    rider_condition = conditioning.sample_riders(10, split="test")
    use_case_condition = conditioning.sample_use_case(10, split="test")
    text_embeddings = conditioning.sample_text(10, split="test")
    condition = {"Rider": rider_condition[idx], "Use Case": use_case_condition[idx], "Text": text_embeddings[idx]}
    return condition

def get_conditions_10k():
    rider_condition = conditioning.sample_riders(10000, split="test")
    use_case_condition = conditioning.sample_use_case(10000, split="test")
    text_embeddings = conditioning.sample_text(10000, split="test")
    conditions = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_embeddings}
    return conditions


def build_text_condition(condition):

    rc = condition["Rider"]
    rc_text = f"Rider Body Dimensions: Upper leg length - {rc[0]}, Lower leg length - {rc[1]}, Arm length - {rc[2]}, Torso length - {rc[3]}, Neck and head length - {rc[4]}, Torso width - {rc[5]}"
    tc = condition["Text"]
    uc = condition["Use Case"]
    #get argmax of uc
    uc = uc.argmax()
    if uc == 0:
        uc_text = "Use Case: Road Biking"
    elif uc == 1:
        uc_text = "Use Case: Mountain Biking"
    elif uc == 2:
        uc_text = "Use Case: Commuting"
    tc_text = "Marketing Description: " + tc
    full_text = f"{rc_text}. {uc_text}. {tc_text}"
    return full_text

cond = get_condition_by_idx(0)
cond_text = build_text_condition(cond)

In [5]:
ds_len = len(data)

text_conditions = []
rider_condition = conditioning.sample_riders(ds_len, split="train", randomize=True)
use_case_condition = conditioning.sample_use_case(ds_len, split="train", randomize=True)
text_embeddings = conditioning.sample_text(ds_len, split="train", randomize=True)
for i in trange(ds_len):
    conditions = {"Rider": rider_condition[i], "Use Case": use_case_condition[i], "Text": text_embeddings[i]}
    text_conditions.append(build_text_condition(conditions))
print("Text conditions built")

with open(os.path.join(filedir, "text_conditions.txt"), "w") as f:
    for item in text_conditions:
        f.write("%s\n" % item)


100%|██████████| 4500/4500 [00:00<00:00, 31774.76it/s]

Text conditions built


In [13]:
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_embeddings}
eval_fns = get_standard_evaluations(device, aesthetics_mode="Text")
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(eval_fns, data_oh.columns)
eval_scores = evaluator(data_tens, condition)
eval_scores_df = pd.DataFrame(eval_scores.cpu().detach().numpy(), columns=requirement_names)
eval_scores_df.to_csv(os.path.join(filedir, "eval_scores.csv"), index=False)

c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
SYSTEM_PROMPT = """
# I will ask you to create some bicycle designs. The bicycle designs are subject to a text prompt, some rider dimensions, and a use case. 
# Each design is defined by 70 variables, which I will describe. Some of these are categorical variables, and I will provide you with the possible values for these variables. Others are continuous.
# The design variables are as follows: 



# """.strip()

In [ ]:
General notes: All lengths are measured in mm. All angles are measured in degrees. General format is: 
'parameter name': [Datatype] Description. 

The 70 variables are described as follows:

'CS textfield': [Continuous] The length of the chain stay tubes.
'BB textfield': [Continuous] Bottom bracket drop, measured as the vertical drop from the rear axle to the center of the bottom bracket. By convention, positive values imply the bottom bracket lies below the axle.
'Stack': [Continuous] The vertical distance from the top of the head tube relative to the bottom bracket.
'Head angle': [Continuous] The angle of the head tube with respect to horizontal, in degrees.
'Head tube length textfield' [Continuous] The length of the head tube.
'Seat stay junction0': [Continuous] The length along the seat tube from the top of the seat tube to the junction with the seat stays. By convention, this is measured to the center of the seat stays. 
'Seat tube length': [Continuous] The length of the seat stay tubes.
'Seat angle': [Continuous] The angle of the seat tube with respect to horizontal.
'DT Length': [Continuous] The length of the down tube.
'FORK0R': [Continuous] Fork offset, measured as the perpendicular distance from the front axle to the head tube axis.
'BB diameter': [Continuous] The diameter of the bottom bracket
'ttd': [Continuous] Top tube outer diameter. 
'csd': [Continuous] Chain stay outer diameter.
'ssd': [Continuous] Seat stay outer diameter.
'dtd': [Continuous] Down tube outer diameter.
'Chain stay position on BB': [Continuous] The distance along the length of the bottom bracket from its edge to the center of the chain stay tubes.
'SSTopZOFFSET': [Continuous] The offset from center plane of the bike of the joints connecting the seat stays to the the seat tube.
'Head tube upper extension2': [Continuous] The length from the top of the head tube to the junction with the top tube. By convention, this is measured to the center of the top tube.
'Seat tube extension2': [Continuous] The length from the top of the seat tube to the junction with the top tube. By convention, this is measured to the center of the top tube.
'Head tube lower extension2': [Continuous] The length from the bottom of the head tube to the junction with the down tube. By convention, this is measured to the center of the down tube.
'SEATSTAYbrdgshift': [Continuous] The distance along the center plane of the bike from the seat stay and seat tube junction to the seat stay bridge, if present on the bike.
'CHAINSTAYbrdgshift': [Continuous] The distance along the center plane of the bike from the outer rim of the bottom bracket to the chain stay bridge, if present on the bike.
'SEATSTAYbrdgdia1': [Continuous] The diameter of the seat stay bridge, if present on the bike.
'CHAINSTAYbrdgdia1': [Continuous] The diameter of the chain stay bridge, if present on the bike.
'SEATSTAYbrdgCheck': [Boolean] A boolean value indicating whether the seat stay bridge is present on the bike.
'CHAINSTAYbrdgCheck': [Boolean] A boolean value indicating whether the chain stay bridge is present on the bike.
'Dropout spacing': [Continuous] The distance between the rear dropouts.
'Wall thickness Bottom Bracket': [Continuous] The tube wall thickness of the bottom bracket.
'Wall thickness Top tube': [Continuous] The tube wall thickness of the top tube.
'Wall thickness Head tube': [Continuous] The tube wall thickness of the head tube.
'Wall thickness Down tube': [Continuous] The tube wall thickness of the down tube.
'Wall thickness Chain stay': [Continuous] The tube wall thickness of the chain stay.
'Wall thickness Seat stay': [Continuous] The tube wall thickness of the seat stay.
'Wall thickness Seat tube': [Continuous] The tube wall thickness of the seat tube.
'Wheel diameter front': [Continuous] The outer diameter of the front wheel.
'RDBSD': [Continuous] The difference between rear wheel outer diameter and bead seat diamater, roughly approximating the tire thickness.
'Wheel diameter rear': [Continuous] The outer diameter of the rear wheel.
'FDBSD': [Continuous] The difference between front wheel outer diameter and bead seat diamater, roughly approximating the tire thickness.
'Display AEROBARS': [Boolean] A boolean value indicating whether the bike has aerobars.
'BB length': [Continuous] The length of the bottom bracket.
'Head tube diameter': [Continuous] Head tube outer diameter.
'Wheel cut': [Continuous] The diameter of the cutout of seat tube for the rear wheel, if using an aerodynamic tube type.
'Seat tube diameter': [Continuous] Seat tube outer diameter.
'bottle SEATTUBE0 show': [Boolean] A boolean value indicating whether the bike has a bottle holder on the seat tube.
'bottle DOWNTUBE0 show': [Boolean] A boolean value indicating whether the bike has a bottle holder on the down tube.
'Front Fender include': [Boolean] A boolean value indicating whether the bike has a front fender.
'Rear Fender include': [Boolean] A boolean value indicating whether the bike has a rear fender.
'BELTorCHAIN': [Boolean] A boolean value indicating whether the bike has a chain (True) as opposed to a belt.
'Number of cogs' [Integer] The number of cogs on the rear wheel.
'Number of chainrings' [Integer] The number of chainrings attached to the crank.
'Display RACK': [Boolean] A boolean value indicating whether the bike has a rack.
'FIRST color R_RGB': [Continuous] The red component of the primary paint color of the bike.
'FIRST color G_RGB': [Continuous] The green component of the primary paint color of the bike.
'FIRST color B_RGB': [Continuous] The blue component of the primary paint color of the bike.
'SPOKES composite front': [Integer] If applicable, the number of composite spokes in the front wheel minus two (a value of 1 is a trispoke wheel).
'SPOKES composite rear': [Integer] If applicable, the number of composite spokes in the rear wheel minus two (a value of 1 is a trispoke wheel).
'SBLADEW front': [Continuous] If applicable, the width of the front wheel composite spokes.
'SBLADEW rear': [Continuous] If applicable, the width of the rear wheel composite spokes.
'Saddle length': [Continuous] The length of the saddle.
'Saddle height': [Continuous] The bertical distance from  the saddle to the bottom bracket.
'Down tube diameter': [Continuous] The diameter of the down tube.
'Seatpost LENGTH': [Continuous] The length of the seat post.
'MATERIAL': [Categorical] The material of the bike frame. Possible values are: 'ALUMINIUM', 'CARBON', 'STEEL', 'TITANIUM', 'BAMBOO', 'OTHER'.
'Head tube type': [Categorical] The style of head tube. Possible values are: '0', '1', '2', '3'. 0 is aerodynamic, while 1 and 2 are standard round tubes with no distinction in this representation scheme. 3 is a tapered head tube.
'RIM_STYLE front': [Categorical] The style of the front rim. Possible values are: 'DISC', 'SPOKED', 'TRISPOKE'. Despite the name, trispoke class implies composite spokes but does not necessarily imply three composite spokes. 
'RIM_STYLE rear': [Categorical] The style of the rear rim. Possible values are: 'DISC', 'SPOKED', 'TRISPOKE'. Despite the name, trispoke class implies composite spokes but does not necessarily imply three composite spokes.
'Handlebar style': [Categorical] The style of the handlebars. Possible values are: '0', '1', '2'. 0 is a drop bar, 1 is a mountain bike bar, 2 is a bullhorn bar.
'Stem kind': [Categorical] The style of stem. Possible values are: '0', '1', '2'. 0 is a stem that features a sharp and immediate angle away from the head tube. 1 is a stem that features a sharp angle some distance away from the head tube. 2 is a stem that features a gradual angle away from the head tube after intially extending in line with the head tube.
'Fork type': [Categorical] The style of fork. Possible values are: '0', '1', '2'. 0 is a standard fork, 1 is a fork with mountain bike shocks, 2 is a time trial bike fork. 
'Seat tube type': [Categorical] The style of seat tube. Possible values are: '0', '1', '2'. 0 is aerodynamic, while 1 and 2 are standard round tubes with no distinction in this representation scheme. 


In [ ]:
# # File paths
# csv1_path = "openai_files/condition.csv"
# csv2_path = "openai_files/data.csv"
# csv3_path = "openai_files/result.csv"
# output_path = "output.csv"

# # Load CSV contents as strings
# with open(csv1_path, "r") as f:
#     csv1_data = f.read()

# with open(csv2_path, "r") as f:
#     csv2_data = f.read()

# # System prompt (model behavior setup)
# SYSTEM_PROMPT = """
# I will ask you to create some bicycle designs. The bicycle designs are subject to a text prompt, some rider dimensions, and a use case. 
# The design consists of 
# """.strip()

# # User instructions (custom logic)
# USER_INSTRUCTIONS = """
# You are provided with two CSV files.

# Please:
# - Join them on the column 'id'
# - Keep only rows where 'status' == 'active'
# - Include only the columns 'id', 'name', and 'score'
# - Return the result as plain CSV with no extra explanation, no Markdown, and no code block formatting
# """.strip()

# # === END CONFIG SECTION ===

# # Prepare the prompt messages
# messages = [
#     {"role": "system", "content": SYSTEM_PROMPT},
#     {
#         "role": "user",
#         "content": (
#             f"CSV File 1:\n{csv1_data}\n\n"
#             f"CSV File 2:\n{csv2_data}\n\n"
#             f"{USER_INSTRUCTIONS}"
#         )
#     }
# ]

# # Call the model
# response = openai.ChatCompletion.create(
#     model="gpt-4",
#     messages=messages,
#     temperature=0,
# )

# # Save the output


In [ ]:
csv_output = response['choices'][0]['message']['content'].strip()

with open(output_path, "w") as f:
    f.write(csv_output)

print(f"✅ Saved generated CSV to: {output_path}")